# Customer Support Intelligence — Python ETL Pipeline

**Notebook:** `02_Python_ETL_Pipeline.ipynb`
**Dataset (input):** `support-tickets.csv` (raw, 4,000 rows / 10 columns)
**Dataset (output):** `support-tickets-clean.csv`

### Scope of this notebook

This is the **second** notebook in the pipeline. It does not re-analyze data quality. This notebook simply **implements** the treatment rules that analysis produced: Extract the raw file, Transform it into the schema the BRD's deliverables call for (applying the exact cleaning rules and IQR fences already decided), run fail-fast structural checks, and Load the result to `support-tickets-clean.csv`.


## 1. Setup

Import the libraries required for the ETL process.

- **Pandas** — data ingestion and transformation
- **NumPy** — numerical operations
- **Pathlib** — file path handling


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

INPUT_FILE = Path("support-tickets.csv")
OUTPUT_FILE = Path("support-tickets-clean.csv")

print("Setup complete.")

Setup complete.


## 2. Extract — Load Raw Customer Support Data

The raw CSV is loaded without modifying any business values. The two timestamp fields (`opened_at`, `resolved_at`) are parsed during ingestion so they can be used safely in date calculations later in the pipeline.

In [ ]:
# Load the raw dataset
try:
    df_raw = pd.read_csv(
        INPUT_FILE,
        parse_dates=["opened_at", "resolved_at"]
    )
except FileNotFoundError:
    # Google Colab fallback
    try:
        from google.colab import files

        print(f"'{INPUT_FILE.name}' was not found.")
        print("Please upload the raw support ticket CSV file.")
        uploaded = files.upload()

        uploaded_name = next(iter(uploaded))
        INPUT_FILE = Path(uploaded_name)

        df_raw = pd.read_csv(
            INPUT_FILE,
            parse_dates=["opened_at", "resolved_at"]
        )
    except ImportError:
        raise FileNotFoundError(
            f"'{INPUT_FILE.name}' was not found. Place the raw CSV next to this "
            "notebook (local Jupyter) or run this cell in Google Colab to upload it."
        )

print(f"Loaded {len(df_raw):,} rows x {df_raw.shape[1]} columns from '{INPUT_FILE.name}'.")

Loaded 4,000 rows x 10 columns from 'support-tickets.csv'.


### Raw data structure

Before transforming the data, inspect the schema so the transformation steps below map cleanly onto the BRD's data dictionary (Section 7.1).

In [ ]:
print("Column data types:")
display(df_raw.dtypes.to_frame("data_type"))

print("\nDataset shape:")
print(f"Rows    : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]:,}")

print("\nSample records:")
display(df_raw.head())

Column data types:


,data_type
ticket_id,object
opened_at,datetime64[ns]
channel,object
queue,object
priority,object
subject,object
first_response_min,float64
resolved_at,datetime64[ns]
reopened,bool
csat,float64



Dataset shape:
Rows    : 4,000
Columns : 10

Sample records:


,ticket_id,opened_at,channel,queue,priority,subject,first_response_min,resolved_at,reopened,csat
0,TK62871,2023-01-01 02:15:00,in-app,onboarding,high,Error when log in after password reset,63.0,2023-01-01 07:46:35,True,NaN
1,TK60343,2023-01-01 05:41:00,email,technical,normal,Question about two-factor authentication,7.0,2023-01-01 09:07:58,False,NaN
2,TK62950,2023-01-01 06:05:00,in-app,billing,urgent,Unable to connecting the Snowflake sync,44.0,2023-01-01 08:14:24,False,NaN
3,TK61014,2023-01-01 06:32:00,chat,integrations,normal,Cannot connecting the Snowflake sync,98.0,NaT,False,NaN
4,TK61679,2023-01-01 08:06:00,chat,onboarding,high,How do I a duplicate charge,14.0,2023-01-01 09:27:09,False,5.0


## 3. Transform — Standardize the Raw Data

Applies the field-by-field treatment decided in the Data Quality Analysis notebook (Section 8 there):

- Standardize column names
- Trim and clean text fields
- Normalize categorical values (lower-case, consistent spelling)
- Convert numeric fields to proper numeric types
- Cast `reopened` to a proper boolean

The original raw DataFrame is kept unchanged. All transformations are applied to a copy, in line with the BRD's non-functional requirement that every metric stay traceable back to the source fields.

In [ ]:
# Create a working copy so the raw extract remains unchanged
df = df_raw.copy()

# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Clean text columns
text_columns = ["ticket_id", "channel", "queue", "priority", "subject"]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# Normalize categorical fields to lower-case for consistent grouping/filtering
categorical_columns = ["channel", "queue", "priority"]

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].str.lower()

# Ensure numeric fields are numeric
df["first_response_min"] = pd.to_numeric(df["first_response_min"], errors="coerce")
df["csat"] = pd.to_numeric(df["csat"], errors="coerce")

# Ensure boolean field is a proper boolean
df["reopened"] = df["reopened"].astype("boolean")

print("Column standardization complete.")
display(df.dtypes.to_frame("data_type"))

Column standardization complete.


,data_type
ticket_id,string[python]
opened_at,datetime64[ns]
channel,string[python]
queue,string[python]
priority,string[python]
subject,string[python]
first_response_min,float64
resolved_at,datetime64[ns]
reopened,boolean
csat,float64


## 4. Transform — Create Derived Fields

Per the treatment rules documented in the Data Quality Analysis notebook, the cleaned dataset carries these derived fields in addition to the raw ones:

| Field | Purpose | BRD reference |
|---|---|---|
| `resolution_min` | Elapsed minutes between `opened_at` and `resolved_at` | FR-05 |
| `is_open` | Ticket has no `resolved_at` yet (still open) | FR-06 |
| `has_csat` | Ticket has a completed CSAT survey | FR-08 |
| `opened_month` | Calendar month the ticket was opened, for trend reporting | FR-01 |
| `frt_sla_flag` | First-response time exceeds the IQR upper fence decided in the DQ analysis | Deliverable: "SLA flag" |
| `resolution_sla_flag` | Resolution time exceeds the IQR upper fence decided in the DQ analysis | Deliverable: "SLA flag" |

The IQR fences are recomputed here using the exact same rule the DQ notebook used (Q3 + 1.5 x IQR) — this is a mechanical application of an already-documented rule, not a new analysis. **No rows are removed or altered because of this flag**; it is carried forward as a column so the dashboard and reporting layer can highlight and filter on it.

In [ ]:
# Resolution time in minutes (NaN for tickets that are still open)
df["resolution_min"] = (
    df["resolved_at"] - df["opened_at"]
).dt.total_seconds() / 60

# Ticket status flags
df["is_open"] = df["resolved_at"].isna()
df["has_csat"] = df["csat"].notna()

# Month field for time-based trend analysis
df["opened_month"] = df["opened_at"].dt.to_period("M").astype("string")

# SLA-style flags using the IQR upper fence (rule decided in the DQ Analysis notebook, Section 6)
def iqr_upper_fence(series):
    values = series.dropna()
    if values.empty:
        return np.nan
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    return q3 + 1.5 * (q3 - q1)

frt_upper = iqr_upper_fence(df["first_response_min"])
resolution_upper = iqr_upper_fence(df["resolution_min"])

df["frt_sla_flag"] = df["first_response_min"].notna() & (df["first_response_min"] > frt_upper)
df["resolution_sla_flag"] = df["resolution_min"].notna() & (df["resolution_min"] > resolution_upper)

display(
    df[[
        "ticket_id", "opened_at", "resolved_at", "resolution_min",
        "is_open", "has_csat", "opened_month", "frt_sla_flag", "resolution_sla_flag",
    ]].head()
)

,ticket_id,opened_at,resolved_at,resolution_min,is_open,has_csat,opened_month,frt_sla_flag,resolution_sla_flag
0,TK62871,2023-01-01 02:15:00,2023-01-01 07:46:35,331.583333,False,False,2023-01,False,False
1,TK60343,2023-01-01 05:41:00,2023-01-01 09:07:58,206.966667,False,False,2023-01,False,False
2,TK62950,2023-01-01 06:05:00,2023-01-01 08:14:24,129.400000,False,False,2023-01,False,False
3,TK61014,2023-01-01 06:32:00,NaT,NaN,True,False,2023-01,False,False
4,TK61679,2023-01-01 08:06:00,2023-01-01 09:27:09,81.150000,False,True,2023-01,False,False


## 5. Fail-Fast Technical Checks

Before exporting, the pipeline re-verifies the structural assumptions the Data Quality Analysis notebook found to hold on the raw extract. These are pass/fail gates meant to catch a *future* broken extract early. They are not a re-analysis of data quality, just a guardrail.

The pipeline does **not** drop rows, impute values, or silently "fix" anything here — a failed check is meant to stop the pipeline and prompt investigation of the source extract, per the BRD's assumption that `support-tickets.csv` is expected to be a complete, accurate export.

In [ ]:
checks = []

# 1. Required columns are present
required_columns = {
    "ticket_id", "opened_at", "channel", "queue", "priority",
    "subject", "first_response_min", "resolved_at", "reopened", "csat",
}
missing_columns = required_columns - set(df.columns)
checks.append(("Required columns present", len(missing_columns) == 0, missing_columns or "-"))

# 2. ticket_id is present on every row and unique
checks.append(("ticket_id has no nulls", df["ticket_id"].notna().all(), int(df["ticket_id"].isna().sum())))
checks.append(("ticket_id is unique", df["ticket_id"].duplicated().sum() == 0, int(df["ticket_id"].duplicated().sum())))

# 3. Categorical fields only contain documented values
expected_values = {
    "channel": {"chat", "email", "in-app", "phone"},
    "queue": {"account", "billing", "integrations", "onboarding", "technical"},
    "priority": {"low", "normal", "high", "urgent"},
}
for col, allowed in expected_values.items():
    unexpected = set(df[col].dropna().unique()) - allowed
    checks.append((f"'{col}' values are all documented", len(unexpected) == 0, unexpected or "-"))

# 4. A resolved ticket cannot be resolved before it was opened
bad_order = df["resolved_at"].notna() & (df["resolved_at"] < df["opened_at"])
checks.append(("resolved_at is never earlier than opened_at", bad_order.sum() == 0, int(bad_order.sum())))

# Report
results = pd.DataFrame(checks, columns=["check", "passed", "detail"])
display(results)

if not results["passed"].all():
    print("\nWARNING: one or more structural checks failed — investigate the source extract "
          "before trusting the cleaned output, and re-run the Data Quality Analysis notebook "
          "to understand the new issue before deciding how to treat it.")
else:
    print("\nAll structural checks passed. Proceeding to export.")

,check,passed,detail
0,Required columns present,True,-
1,ticket_id has no nulls,True,0
2,ticket_id is unique,True,0
3,'channel' values are all documented,True,-
4,'queue' values are all documented,True,-
5,'priority' values are all documented,True,-
6,resolved_at is never earlier than opened_at,True,0



All structural checks passed. Proceeding to export.


## 6. Load — Export the Cleaned Dataset

The transformed dataset is exported as `support-tickets-clean.csv`. This file is the single handoff artifact for:
- Exploratory Data Analysis
- SQL analysis
- Power BI data modeling and dashboard build

The pipeline does **not** delete operationally meaningful outliers or open tickets — per the BRD's non-functional requirement, incomplete records must stay visible and labeled, not silently dropped.

In [ ]:
# Export the transformed data
df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved cleaned dataset to: {OUTPUT_FILE.resolve()}")
print(f"Output shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nFinal columns:")
print(list(df.columns))

Saved cleaned dataset to: /content/support-tickets-clean.csv
Output shape: 4,000 rows x 16 columns

Final columns:
['ticket_id', 'opened_at', 'channel', 'queue', 'priority', 'subject', 'first_response_min', 'resolved_at', 'reopened', 'csat', 'resolution_min', 'is_open', 'has_csat', 'opened_month', 'frt_sla_flag', 'resolution_sla_flag']


## 7. Pipeline Summary

```text
Raw Customer Support CSV (support-tickets.csv)
          |
   [ Data quality rules decided in 01_Support_Tickets_Data_Quality_Analysis.ipynb ]
          |
       EXTRACT
    pandas.read_csv()
          |
      TRANSFORM
  - Standardize column names & text fields
  - Normalize categorical values
  - Convert data types
  - Derive resolution_min, is_open, has_csat, opened_month
  - Derive frt_sla_flag / resolution_sla_flag (IQR rule from the DQ analysis)
          |
   FAIL-FAST CHECKS
  - Required columns present
  - ticket_id: no nulls, unique
  - Categorical values documented
  - Date ordering valid
          |
         LOAD
   support-tickets-clean.csv
```
